# CMU research metrics — notebook starter (Fizzy #4651, A4)

Reproduce the research metrics locally with the **same pinned package** the
service endpoint uses, then call the research API with **your own personal
`crt_` token**.

> **Security boundary.** This notebook never uses a Supabase service-role key.
> The Vercel Python metrics function has no database access; all approved data
> is loaded by the Supabase Edge Function, which signs a sanitized payload to
> the Python service. Your token's role decides what you may do:
> **Viewer** can read/export; **Researcher** can also import/run; **Admin** can
> manage settings and members.

In [ ]:
import os, pathlib, sys

# Import the pinned package straight from the repo (no duplicate formulas).
# Resolve it robustly so the notebook works from notebooks/ or the repo root.
CANDIDATES = [
    os.environ.get("RESEARCH_METRICS_PYTHONPATH"),
    str(pathlib.Path.cwd() / "python"),
    str(pathlib.Path.cwd().parent / "python"),
    str(pathlib.Path.cwd().parent.parent / "python"),
]
for candidate in CANDIDATES:
    if candidate and (pathlib.Path(candidate) / "research_metrics" / "__init__.py").exists():
        sys.path.insert(0, candidate)
        break
else:
    raise RuntimeError("Could not locate research_metrics (set RESEARCH_METRICS_PYTHONPATH).")

import research_metrics

print("pinned package version:", research_metrics.__version__)
print("public metrics:", [n for n in research_metrics.__all__ if not n.startswith("__")])

## 1. Configure your personal token

Issue a token from the research API (`issue_token`) — it is a `crt_…` string.
Put it in the `RESEARCH_TOKEN` environment variable; never paste a
service-role key here.

In [ ]:
import json, os, urllib.request

RESEARCH_API_URL = os.environ.get(
    "RESEARCH_API_URL",
    "https://<project-ref>.supabase.co/functions/v1/fn_research_metrics",
)
RESEARCH_TOKEN = os.environ.get("RESEARCH_TOKEN", "")
assert "service_role" not in RESEARCH_TOKEN.lower(), "personal crt_ token only"


def call_research_api(action, **payload):
    """Call the authenticated research API with your personal token."""
    if not RESEARCH_TOKEN:
        raise RuntimeError("Set RESEARCH_TOKEN to your personal crt_ token.")
    body = json.dumps({"action": action, **payload}).encode()
    request = urllib.request.Request(
        RESEARCH_API_URL,
        data=body,
        method="POST",
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {RESEARCH_TOKEN}",
        },
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        return json.loads(response.read().decode())

## 2. Reproduce the metrics from the fixed synthetic fixture

This is the same fixture the endpoint contract test uses. Because both paths
call `research_metrics.compute_metrics`, the numbers are identical by
construction.

In [ ]:
SYNTHETIC_PAYLOAD = {
    "agreement": {
        "human": [4, 3, 5, 2, 4, 3, 4, 5, 2, 3],
        "ai": [4, 3, 4, 2, 5, 3, 3, 5, 2, 4],
    },
    "repeats": [[4, 4, 4], [3, 3, 4], [5, 5, 5], [2, 2, 2], [4, 4, 3], [3, 4, 3]],
    "paired": {
        "a": [3, 3, 4, 2, 3, 4],
        "b": [4, 3, 4, 3, 4, 4],
        "groups": ["clarity", "clarity", "clarity", "ownership", "ownership", "ownership"],
    },
    "versions": {
        "v1": [4, 3, 4, 2, 5, 3, 3, 5, 2, 4],
        "v2": [4, 3, 5, 2, 5, 4, 3, 5, 3, 4],
        "human": [4, 3, 5, 2, 4, 3, 4, 5, 2, 3],
    },
}

local_report = research_metrics.build_report(
    research_metrics.compute_metrics(SYNTHETIC_PAYLOAD),
    warn_threshold=10,
    thresholds=None,  # no study thresholds => no threshold flags
)
local_report

## 3. Fetch the same metrics for a real completed run

Replace `run_id` with a completed A3 run UUID. The API loads the approved
frozen data, computes the metrics through the pinned package, and returns the
descriptive report (with `n` and 95% CIs).

In [ ]:
# run_id = "00000000-0000-4000-8000-000000000000"
# api_report = call_research_api("metrics", run_id=run_id)
# api_report["report"]

# Safe export: approved item text, human ratings, scores, safe explanations,
# confidence, metrics and a redacted provenance manifest.
# export = call_research_api("export", run_id=run_id)
# export["items"][0]

## 4. Confirm your role's permissions

A **Viewer** token can read and export but cannot import or run; a **Researcher**
can also import/run; an **Admin** can additionally manage settings/members. The
API enforces this — the notebook cannot widen it.

In [ ]:
# me = call_research_api("whoami")
# me["role"], me["capabilities"]
# assert me["capabilities"]["export"] is True
# viewer tokens: me["capabilities"]["run"] is False and me["capabilities"]["import"] is False